In [ ]:
# ==============================================================================
# Cell 1: KAGGLE & HARDWARE ENVIRONMENT CHECK — FIXED
# ==============================================================================
import sys
import torch

MIN_VRAM_GB = 10.0  # Yêu cầu tối thiểu cho ViViT-B batch=4

print("=" * 70)
print("KAGGLE & HARDWARE ENVIRONMENT CHECK")
print("=" * 70)
print(f"  Python  : {sys.version.split()[0]}")
print(f"  PyTorch : {torch.__version__}")
print(f"  CUDA    : {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")
print(f"  cuDNN   : {torch.backends.cudnn.version() if torch.cuda.is_available() else 'N/A'}")

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU KHÔNG KHẢ DỤNG. Pipeline yêu cầu GPU (T4 / P100 / A100).\n"
        "→ Trên Kaggle: Settings → Accelerator → GPU T4 x2"
    )

print(f"  GPU Count: {torch.cuda.device_count()}")

all_gpus_ok = True
for i in range(torch.cuda.device_count()):
    props = torch.cuda.get_device_properties(i)
    vram_gb = props.total_memory / 1024**3
    status = "✅" if vram_gb >= MIN_VRAM_GB else "⚠️ VRAM thấp"
    print(f"  [GPU {i}] {props.name} | VRAM: {vram_gb:.1f} GB | {status}")
    if vram_gb < MIN_VRAM_GB:
        all_gpus_ok = False

if not all_gpus_ok:
    print(f"\n  ⚠️ Ít nhất 1 GPU có VRAM < {MIN_VRAM_GB} GB.")
    print(f"     Cân nhắc giảm TRAIN_BATCH_SIZE hoặc bật gradient_checkpointing.")

print("=" * 70)

In [ ]:
# Verify cài đặt thành công
import importlib
importlib.invalidate_caches()  # ← Flush cache sau pip install

required = {
    "transformers": "4.48",
    "accelerate": "0.34",
    "sklearn": None,       # Chỉ check import
    "cv2": "4.8",
    "matplotlib": "3.7",
}

for pkg, min_ver in required.items():
    try:
        mod = importlib.import_module(pkg)
        ver = getattr(mod, "__version__", "unknown")
        
        if min_ver and ver != "unknown":
            from packaging.version import Version
            if Version(ver) < Version(min_ver):
                print(f"  ⚠️ {pkg}: {ver} (cần >= {min_ver}) — restart kernel!")
                continue
        
        print(f"  ✅ {pkg}: {ver}")
    except ImportError:
        print(f"  ❌ {pkg}: FAILED — cần cài lại thủ công!")

In [ ]:
# ==============================================================================
# Cell 3: IMPORTS & GLOBAL SEEDING — FIXED
# ==============================================================================
import os
import re
import sys
import json
import random
import hashlib
import shutil
import logging
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report

import torch
import torch.nn as nn
from torch.utils.data import Dataset

import transformers
from transformers import (
    Trainer,
    TrainingArguments,
    VivitModel,
    VivitImageProcessor,
    EarlyStoppingCallback,
)
from transformers.modeling_outputs import ImageClassifierOutput
from transformers.modeling_utils import unwrap_model
from transformers.trainer_utils import get_last_checkpoint


# ── Reproducibility ──
def set_seed(seed: int):
    """
    Full reproducibility: Python + NumPy + PyTorch + cuDNN + CUBLAS.
    Gọi hàm này TẠI Cell 4 sau khi định nghĩa SEED, không gọi ở đây.
    """
    os.environ["PYTHONHASHSEED"] = str(seed)
    os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"   # Required cho CUDA >= 10.2

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    # PyTorch 2.0+: cover TẤT CẢ non-deterministic ops (scatter, index_add, etc.)
    torch.use_deterministic_algorithms(True, warn_only=True)
    torch.backends.cudnn.benchmark = False


# ── Version log cho reproducibility ──
print(f"{'='*70}")
print(f"ENVIRONMENT & DEPENDENCIES")
print(f"{'='*70}")
print(f"  Python       : {sys.version.split()[0]}")
print(f"  PyTorch      : {torch.__version__}")
print(f"  Transformers : {transformers.__version__}")
print(f"  NumPy        : {np.__version__}")
print(f"  OpenCV       : {cv2.__version__}")
print(f"  scikit-learn : {__import__('sklearn').__version__}")
print(f"  CUDA         : {torch.version.cuda if torch.cuda.is_available() else 'N/A'}")
print(f"  cuDNN        : {torch.backends.cudnn.version() if torch.cuda.is_available() else 'N/A'}")
print(f"{'='*70}")

In [ ]:
# ==============================================================================
# Cell 4: GLOBAL CONFIGURATION & HYPERPARAMETERS — FIXED
# ==============================================================================

# ── Paths ──
DATASET_ROOT = "/kaggle/input/datasets/doannh/dataset"
PERSISTENT_RUN_ROOT = "/kaggle/working/EduViolence_ViViT_runs"
MODEL_OUTPUT_ROOT = os.path.join(PERSISTENT_RUN_ROOT, "checkpoints")
RESULT_OUTPUT_ROOT = os.path.join(PERSISTENT_RUN_ROOT, "results")

# ── Validate paths ──
if not os.path.isdir(DATASET_ROOT):
    raise FileNotFoundError(
        f"DATASET_ROOT không tồn tại: {DATASET_ROOT}\n"
        f"Kiểm tra: đã add dataset vào notebook Kaggle chưa?"
    )

os.makedirs(MODEL_OUTPUT_ROOT, exist_ok=True)
os.makedirs(RESULT_OUTPUT_ROOT, exist_ok=True)

# ── ViViT Model Config ──
MODEL_NAME = "google/vivit-b-16x2-kinetics400"
NUM_FRAMES = 16

# ── Training Hyperparameters ──
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
GRAD_ACCUM_STEPS = 2
NUM_EPOCHS = 30
LEARNING_RATE = 3e-5
WEIGHT_DECAY = 0.05
WARMUP_RATIO = 0.15
EARLY_STOP_PATIENCE = 3
NUM_WORKERS = 2
SEED = 42

# ── ✅ FIX: Gọi set_seed ngay sau khi định nghĩa SEED ──
set_seed(SEED)

# ── Eval & Save frequency ──
SAVE_STEPS = 50
EVAL_STEPS = 50

# ── Tính toán và hiển thị effective config ──
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 1
EFFECTIVE_BATCH_SIZE = TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS * N_GPUS

print(f"{'='*70}")
print(f"GLOBAL CONFIGURATION")
print(f"{'='*70}")
print(f"  Dataset        : {DATASET_ROOT}")
print(f"  Model          : {MODEL_NAME}")
print(f"  Num Frames     : {NUM_FRAMES}")
print(f"  GPUs detected  : {N_GPUS}")
print(f"  Train Batch/GPU: {TRAIN_BATCH_SIZE}")
print(f"  Eval Batch/GPU : {EVAL_BATCH_SIZE}")
print(f"  Grad Accum     : {GRAD_ACCUM_STEPS}")
print(f"  Effective Batch: {TRAIN_BATCH_SIZE} × {GRAD_ACCUM_STEPS} × {N_GPUS} = {EFFECTIVE_BATCH_SIZE}")
print(f"  Learning Rate  : {LEARNING_RATE}")
print(f"  Weight Decay   : {WEIGHT_DECAY}")
print(f"  Warmup Ratio   : {WARMUP_RATIO}")
print(f"  Epochs (max)   : {NUM_EPOCHS}")
print(f"  Early Stop Pat.: {EARLY_STOP_PATIENCE}")
print(f"  Eval/Save Steps: {EVAL_STEPS}")
print(f"  Seed           : {SEED}")
print(f"  Outputs        : {PERSISTENT_RUN_ROOT}")

# ── Cảnh báo nếu effective batch size khác kỳ vọng ──
EXPECTED_EFF_BATCH = 32
if EFFECTIVE_BATCH_SIZE != EXPECTED_EFF_BATCH:
    import math
    print(f"\n  ⚠️  Effective batch = {EFFECTIVE_BATCH_SIZE} (kỳ vọng {EXPECTED_EFF_BATCH}).")
    print(f"      LR được tune cho batch={EXPECTED_EFF_BATCH}.")
    # ✅ FIX: Sqrt Scaling Rule cho AdamW (adaptive optimizer)
    suggested_lr = LEARNING_RATE * math.sqrt(EFFECTIVE_BATCH_SIZE / EXPECTED_EFF_BATCH)
    print(f"      Theo Sqrt Scaling Rule (AdamW), nên điều chỉnh LR → {suggested_lr:.2e}")

# ── Disk space check ──
disk_usage = shutil.disk_usage("/kaggle/working")
free_gb = disk_usage.free / 1024**3
print(f"\n  Disk free: {free_gb:.1f} GB", end="")
if free_gb < 5.0:
    print(f" ⚠️ Thấp! Cần ít nhất ~3-5 GB cho checkpoints + results.")
else:
    print(f" ✅")

print(f"{'='*70}")

In [ ]:
# ==============================================================================
# Cell 5: DATASET AUDIT & DUPLICATE CLEANING (SHA-256) — FIXED
# ==============================================================================
VIDEO_EXTS = {".mp4", ".avi", ".mov", ".mkv", ".webm"}

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1: QUÉT VÀ PARSE CẤU TRÚC THƯ MỤC (AN TOÀN CHIỀU SÂU)
# ─────────────────────────────────────────────────────────────────────────────
records = []
skipped_files = []

for root, dirs, files in os.walk(DATASET_ROOT):
    for f in files:
        if Path(f).suffix.lower() in VIDEO_EXTS:
            fp = os.path.join(root, f)
            rel = Path(fp).relative_to(DATASET_ROOT)
            rel_parts = rel.parts  # Kỳ vọng: (<Group>, <Class>, <Source>, <filename>)

            if len(rel_parts) != 4:
                skipped_files.append({
                    "filepath": fp,
                    "depth": len(rel_parts),
                    "relative_path": str(rel)
                })
                continue

            group, cls, source, _ = rel_parts
            records.append({
                "filepath": fp,
                "group": group,
                "class_name": cls,
                "source": source
            })

df_raw = pd.DataFrame(records)

# ── FIX: Guard empty DataFrame trước khi truy cập columns ──
if df_raw.empty:
    raise ValueError(
        f"Không tìm thấy video nào trong {DATASET_ROOT}.\n"
        f"Kiểm tra cấu trúc: <Group>/<Class>/<Source>/<video_file>\n"
        f"Extensions hỗ trợ: {VIDEO_EXTS}"
    )

print(f"Tổng số video hợp lệ quét được: {len(df_raw)}")
if skipped_files:
    print(f"⚠️  Đã bỏ qua {len(skipped_files)} file có cấu trúc thư mục không khớp:")
    for sf in skipped_files[:5]:
        print(f"   - [{sf['depth']} levels] {sf['relative_path']}")
    if len(skipped_files) > 5:
        print(f"   ... và {len(skipped_files) - 5} file khác.")

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: PHÂN NHÓM THEO KÍCH THƯỚC → HASH SHA-256
# ─────────────────────────────────────────────────────────────────────────────
def get_sha256(filepath):
    """Hash nội dung file theo chunk 1 MB để tránh nạp toàn bộ vào RAM."""
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(1024 * 1024):
            h.update(chunk)
    return h.hexdigest()


# 2a. Nhóm theo file size (O(1) — tránh hash toàn bộ dataset)
size_groups = defaultdict(list)
for p in df_raw["filepath"]:
    size_groups[os.path.getsize(p)].append(p)

# 2b. Chỉ hash các nhóm có >= 2 file trùng kích thước
hash_groups = defaultdict(list)
files_needing_hash = sum(len(ps) for ps in size_groups.values() if len(ps) > 1)
print(f"Số file cần hash SHA-256 (trùng kích thước): {files_needing_hash} / {len(df_raw)}")

for size, paths in size_groups.items():
    if len(paths) > 1:
        for p in paths:
            hash_groups[get_sha256(p)].append(p)

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3: XỬ LÝ TRÙNG LẶP — MAJORITY VOTE & DETERMINISTIC SELECTION
# ─────────────────────────────────────────────────────────────────────────────
files_to_remove = set()
dedup_log = []

for digest, paths in hash_groups.items():
    if len(paths) <= 1:
        continue

    sub_df = df_raw[df_raw["filepath"].isin(paths)]
    unique_classes = sub_df["class_name"].unique()

    # FIX: Same-hash = byte-for-byte identical → resolution check vô nghĩa.
    #      Chọn bản giữ lại theo sorted path (deterministic, reproducible).
    sorted_paths = sorted(paths)

    if len(unique_classes) > 1:
        # ── CROSS-CLASS CONFLICT: Majority Vote ──
        class_counts = sub_df["class_name"].value_counts()
        majority_class = class_counts.index[0]
        majority_count = class_counts.iloc[0]

        # Nếu hòa phiếu (tie) → xóa toàn bộ vì không thể phân xử
        is_tie = (class_counts == majority_count).sum() > 1
        if is_tie:
            files_to_remove.update(paths)
            dedup_log.append({
                "sha256_short": digest[:12],
                "action": "REMOVE_ALL (tie vote)",
                "classes_involved": str(list(unique_classes)),
                "num_copies": len(paths),
                "kept": "None"
            })
        else:
            # Xóa toàn bộ bản thiểu số
            minority_paths = sub_df[sub_df["class_name"] != majority_class]["filepath"].tolist()
            files_to_remove.update(minority_paths)

            # Giữ 1 bản majority (sorted path → deterministic)
            majority_paths = sorted(
                sub_df[sub_df["class_name"] == majority_class]["filepath"].tolist()
            )
            kept_file = majority_paths[0]
            for p in majority_paths[1:]:
                files_to_remove.add(p)

            dedup_log.append({
                "sha256_short": digest[:12],
                "action": f"MAJORITY_VOTE -> keep '{majority_class}'",
                "classes_involved": str(list(unique_classes)),
                "num_copies": len(paths),
                "kept": os.path.basename(kept_file)
            })
    else:
        # ── SAME-CLASS DUPLICATE: Giữ 1 bản (sorted path → deterministic) ──
        kept_file = sorted_paths[0]
        for p in sorted_paths[1:]:
            files_to_remove.add(p)

        dedup_log.append({
            "sha256_short": digest[:12],
            "action": f"SAME_CLASS dedup ('{unique_classes[0]}')",
            "classes_involved": str(list(unique_classes)),
            "num_copies": len(paths),
            "kept": os.path.basename(kept_file)
        })

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 4: ÁP DỤNG BỘ LỌC VÀ KIỂM TRA TÍNH TOÀN VẸN
# ─────────────────────────────────────────────────────────────────────────────
df_clean = df_raw[~df_raw["filepath"].isin(files_to_remove)].copy().reset_index(drop=True)

n_removed = len(files_to_remove)
n_same_class = sum(1 for d in dedup_log if "SAME_CLASS" in d["action"])
n_cross_class = sum(1 for d in dedup_log if "MAJORITY_VOTE" in d["action"] or "REMOVE_ALL" in d["action"])

print(f"\n{'='*70}")
print(f"KẾT QUẢ KHỬ TRÙNG LẶP")
print(f"{'='*70}")
print(f"  Tổng file loại bỏ      : {n_removed}")
print(f"  ├─ Same-class duplicates: {n_same_class} nhóm")
print(f"  └─ Cross-class conflicts: {n_cross_class} nhóm")
print(f"  Số mẫu sạch còn lại    : {len(df_clean)}")

MIN_EXPECTED = 2000
if len(df_clean) < MIN_EXPECTED:
    raise ValueError(
        f"Dataset quá nhỏ sau dedup: {len(df_clean)} < {MIN_EXPECTED}. "
        f"Kiểm tra lại DATASET_ROOT hoặc cấu trúc thư mục."
    )

# Lưu log dedup để truy vết
dedup_log_df = pd.DataFrame(dedup_log)
dedup_log_path = os.path.join(RESULT_OUTPUT_ROOT, "dedup_audit_log.csv")
dedup_log_df.to_csv(dedup_log_path, index=False)
print(f"  Log chi tiết dedup lưu tại: {dedup_log_path}")

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 5: TRỰC QUAN HÓA QUÁ TRÌNH LÀM SẠCH DỮ LIỆU
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(20, 14))
fig.suptitle("DATASET AUDIT & DUPLICATE CLEANING REPORT",
             fontsize=16, fontweight="bold", y=0.98)

# ── [1] Phân phối TRƯỚC khi dedup (theo Group + Class) ──
ax1 = axes[0, 0]
raw_class_counts = df_raw.groupby(["group", "class_name"]).size().reset_index(name="count")
groups = sorted(raw_class_counts["group"].unique())
cmap = plt.cm.get_cmap("Set2", max(len(groups), 2))
group_colors = {g: cmap(i) for i, g in enumerate(groups)}

raw_class_counts = raw_class_counts.sort_values(["group", "count"], ascending=[True, False])
bar_colors = [group_colors[g] for g in raw_class_counts["group"]]
bars1 = ax1.barh(raw_class_counts["class_name"], raw_class_counts["count"],
                 color=bar_colors, edgecolor="white")
ax1.set_xlabel("Số lượng video")
ax1.set_title("① Phân phối TRƯỚC dedup (theo Group)", fontsize=12, fontweight="bold")
ax1.invert_yaxis()

for bar, val in zip(bars1, raw_class_counts["count"]):
    ax1.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, str(val),
             va="center", fontsize=9, fontweight="bold")

legend_patches = [Patch(facecolor=group_colors[g], label=g) for g in groups]
ax1.legend(handles=legend_patches, title="Group", loc="lower right", fontsize=9)

# ── [2] So sánh TRƯỚC / SAU dedup với delta ──
ax2 = axes[0, 1]
before_counts = df_raw["class_name"].value_counts().sort_index()
after_counts = df_clean["class_name"].value_counts().sort_index()

all_classes = sorted(set(before_counts.index) | set(after_counts.index))
before_vals = [before_counts.get(c, 0) for c in all_classes]
after_vals = [after_counts.get(c, 0) for c in all_classes]

x_pos = np.arange(len(all_classes))
bar_width = 0.38
ax2.barh(x_pos - bar_width / 2, before_vals, bar_width,
         label="Trước dedup", color="#bdd7ee", edgecolor="white")
ax2.barh(x_pos + bar_width / 2, after_vals, bar_width,
         label="Sau dedup", color="#2e75b6", edgecolor="white")
ax2.set_yticks(x_pos)
ax2.set_yticklabels(all_classes)
ax2.set_xlabel("Số lượng video")
ax2.set_title("② So sánh TRƯỚC / SAU dedup", fontsize=12, fontweight="bold")
ax2.legend(fontsize=10, loc="lower right")
ax2.invert_yaxis()

for i, (bv, av) in enumerate(zip(before_vals, after_vals)):
    diff = bv - av
    if diff > 0:
        ax2.text(max(bv, av) + 1, i, f"-{diff}",
                 va="center", fontsize=9, color="red", fontweight="bold")

# ── [3] Phân phối nguồn video (Source Breakdown) ──
ax3 = axes[1, 0]
source_counts = df_clean.groupby(["group", "source"]).size().reset_index(name="count")
source_counts = source_counts.sort_values("count", ascending=True)
src_colors = [group_colors.get(g, "#999999") for g in source_counts["group"]]

bars3 = ax3.barh(source_counts["source"], source_counts["count"],
                 color=src_colors, edgecolor="white")
ax3.set_xlabel("Số lượng video (sau dedup)")
ax3.set_title("③ Phân phối theo nguồn (Source)", fontsize=12, fontweight="bold")

for bar, val in zip(bars3, source_counts["count"]):
    ax3.text(bar.get_width() + 1, bar.get_y() + bar.get_height() / 2, str(val),
             va="center", fontsize=9, fontweight="bold")

# ── [4] Tổng hợp kết quả Dedup (Pie + Annotation) ──
ax4 = axes[1, 1]

kept_count = len(df_clean)
removed_same = sum(d["num_copies"] - 1 for d in dedup_log if "SAME_CLASS" in d["action"])
removed_cross = max(n_removed - removed_same, 0)

pie_data = [
    ("Giữ lại (Clean)", kept_count, "#2e75b6"),
    ("Loại — Same-Class Dup", removed_same, "#ffc000"),
    ("Loại — Cross-Class", removed_cross, "#ff4444"),
]
pie_data = [(l, s, c) for l, s, c in pie_data if s > 0]

if pie_data:
    p_labels, p_sizes, p_colors = zip(*pie_data)
else:
    p_labels, p_sizes, p_colors = ["Giữ lại (Clean)"], [kept_count], ["#2e75b6"]

ax4.pie(
    p_sizes, labels=p_labels, colors=p_colors,
    autopct=lambda pct: f"{pct:.1f}%\n({int(round(pct / 100 * sum(p_sizes)))})",
    startangle=90, textprops={"fontsize": 10}
)
ax4.set_title("④ Tổng hợp Dedup Pipeline", fontsize=12, fontweight="bold")

summary_text = (
    f"Raw: {len(df_raw)} videos\n"
    f"Clean: {len(df_clean)} videos\n"
    f"Removed: {n_removed} ({n_removed / max(len(df_raw), 1) * 100:.1f}%)\n"
    f"Dedup groups: {len(dedup_log)}"
)
ax4.text(-1.6, -1.3, summary_text, fontsize=10, verticalalignment="top",
         bbox=dict(boxstyle="round,pad=0.5", facecolor="#f0f0f0", edgecolor="#cccccc"))

plt.tight_layout(rect=[0, 0, 1, 0.95])

audit_fig_path = os.path.join(RESULT_OUTPUT_ROOT, "dataset_audit_report.png")
plt.savefig(audit_fig_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"\n✅ Biểu đồ audit đã lưu tại: {audit_fig_path}")
print(f"✅ Dataset sạch sẵn sàng cho bước tiếp theo: {len(df_clean)} mẫu")

In [ ]:
# ==============================================================================
# Cell 6: STRATIFIED DATA SPLIT (70% / 15% / 15%) — FIXED
# ==============================================================================
CLASSES = sorted(df_clean["class_name"].unique())
label2id = {c: i for i, c in enumerate(CLASSES)}
id2label = {i: c for i, c in enumerate(CLASSES)}

df_clean["label_id"] = df_clean["class_name"].map(label2id)

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1: KIỂM TRA ĐIỀU KIỆN TIÊN QUYẾT CHO STRATIFIED SPLIT
# ─────────────────────────────────────────────────────────────────────────────
min_required_per_class = 7  # Tối thiểu để chia 3 split có đại diện
class_dist = df_clean["label_id"].value_counts().sort_index()

tiny_classes = class_dist[class_dist < min_required_per_class]
if len(tiny_classes) > 0:
    print("⚠️ CÁC CLASS QUÁ NHỎ ĐỂ STRATIFIED SPLIT:")
    for lid, cnt in tiny_classes.items():
        print(f"   - {id2label[lid]}: {cnt} mẫu (cần >= {min_required_per_class})")
    raise ValueError(
        f"{len(tiny_classes)} class không đủ mẫu. "
        f"Gộp hoặc loại class nhỏ trước khi split."
    )

print(f"✅ Tất cả {len(CLASSES)} class đều >= {min_required_per_class} mẫu. "
      f"Min = {class_dist.min()} ({id2label[class_dist.idxmin()]})")

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2: TWO-STAGE STRATIFIED SPLIT
# ─────────────────────────────────────────────────────────────────────────────
# Split 1: Tách 15% Test
train_val_df, test_df = train_test_split(
    df_clean, test_size=0.15, random_state=SEED, stratify=df_clean["label_id"]
)

# Split 2: Tách 15% Val trên tổng thể (= 0.15 / 0.85 trên tập train_val)
val_ratio_adjusted = 0.15 / 0.85
train_df, val_df = train_test_split(
    train_val_df, test_size=val_ratio_adjusted, random_state=SEED,
    stratify=train_val_df["label_id"]
)

train_df = train_df.copy().assign(split="train")
val_df = val_df.copy().assign(split="val")
test_df = test_df.copy().assign(split="test")

split_df = pd.concat([train_df, val_df, test_df], axis=0).reset_index(drop=True)

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3: KIỂM CHỨNG CHẤT LƯỢNG SPLIT
# ─────────────────────────────────────────────────────────────────────────────
total = len(df_clean)
actual_ratios = {
    "train": len(train_df) / total,
    "val": len(val_df) / total,
    "test": len(test_df) / total,
}

print(f"\n{'='*70}")
print(f"SPLIT VERIFICATION")
print(f"{'='*70}")
print(f"  Train: {len(train_df):>5} ({actual_ratios['train']:.2%})  |  "
      f"Val: {len(val_df):>5} ({actual_ratios['val']:.2%})  |  "
      f"Test: {len(test_df):>5} ({actual_ratios['test']:.2%})")

# Kiểm tra mỗi class có mặt trong cả 3 split
train_classes = set(train_df["label_id"].unique())
val_classes = set(val_df["label_id"].unique())
test_classes = set(test_df["label_id"].unique())
all_label_ids = set(range(len(CLASSES)))

missing_train = all_label_ids - train_classes
missing_val = all_label_ids - val_classes
missing_test = all_label_ids - test_classes

# FIX: Nâng warning thành error — class thiếu sẽ gây metric misleading ở Cell 13-16
if missing_train or missing_val or missing_test:
    msg_parts = []
    if missing_train:
        msg_parts.append(f"  Train thiếu: {[id2label[i] for i in missing_train]}")
    if missing_val:
        msg_parts.append(f"  Val thiếu:   {[id2label[i] for i in missing_val]}")
    if missing_test:
        msg_parts.append(f"  Test thiếu:  {[id2label[i] for i in missing_test]}")
    raise ValueError(
        "CLASS BỊ THIẾU TRONG SPLIT — metric downstream sẽ misleading!\n"
        + "\n".join(msg_parts) + "\n"
        "→ Tăng min_required_per_class hoặc gộp class nhỏ."
    )
else:
    print("  ✅ Tất cả class đều có mặt trong cả 3 split.")

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 4: LƯU METADATA & LABEL MAP (FIX JSON KEY TYPE)
# ─────────────────────────────────────────────────────────────────────────────
split_df.to_csv(os.path.join(RESULT_OUTPUT_ROOT, "dataset_split_metadata.csv"), index=False)

# Fix: serialize id2label với string key để JSON roundtrip an toàn
id2label_serializable = {str(k): v for k, v in id2label.items()}

with open(os.path.join(RESULT_OUTPUT_ROOT, "label_map.json"), "w", encoding="utf-8") as f:
    json.dump({
        "label2id": label2id,
        "id2label": id2label_serializable
    }, f, indent=2, ensure_ascii=False)

print(f"  Label map đã lưu (id2label keys = string, tương thích JSON roundtrip)")

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 5: TRỰC QUAN HÓA PHÂN PHỐI SPLIT
# ─────────────────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(22, 7))
fig.suptitle("STRATIFIED SPLIT DISTRIBUTION (70% / 15% / 15%)",
             fontsize=15, fontweight="bold", y=1.02)

split_colors = {"train": "#2e75b6", "val": "#ed7d31", "test": "#70ad47"}
split_names = ["train", "val", "test"]
split_dfs = {"train": train_df, "val": val_df, "test": test_df}

for ax, split_name in zip(axes, split_names):
    sdf = split_dfs[split_name]
    counts = sdf["class_name"].value_counts().reindex(CLASSES, fill_value=0)

    bars = ax.barh(CLASSES, counts.values, color=split_colors[split_name],
                   edgecolor="white", alpha=0.85)
    ax.set_xlabel("Số lượng video")
    ax.set_title(f"{split_name.upper()} ({len(sdf)} mẫu — {actual_ratios[split_name]:.1%})",
                 fontsize=12, fontweight="bold")
    ax.invert_yaxis()

    for bar, val in zip(bars, counts.values):
        ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height() / 2,
                str(val), va="center", fontsize=9, fontweight="bold")

    # Đường trung bình
    mean_count = counts.mean()
    ax.axvline(mean_count, color="red", linestyle="--", linewidth=1, alpha=0.7)
    ax.text(mean_count + 0.5, len(CLASSES) - 0.5, f"avg={mean_count:.0f}",
            fontsize=8, color="red")

plt.tight_layout()

split_fig_path = os.path.join(RESULT_OUTPUT_ROOT, "split_distribution.png")
plt.savefig(split_fig_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

# ── Biểu đồ phụ: So sánh tỷ lệ class giữa 3 split (stacked %) ──
fig2, ax2 = plt.subplots(figsize=(14, 6))

split_proportions = {}
for split_name in split_names:
    sdf = split_dfs[split_name]
    props = sdf["class_name"].value_counts(normalize=True).reindex(CLASSES, fill_value=0)
    split_proportions[split_name] = props.values

x_pos = np.arange(len(CLASSES))
bar_w = 0.25

for i, split_name in enumerate(split_names):
    ax2.bar(x_pos + i * bar_w, split_proportions[split_name] * 100,
            bar_w, label=split_name.upper(), color=split_colors[split_name],
            edgecolor="white", alpha=0.85)

ax2.set_xticks(x_pos + bar_w)
ax2.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=9)
ax2.set_ylabel("Tỷ lệ trong split (%)")
ax2.set_title("Kiểm chứng Stratification: Tỷ lệ class nên đồng nhất giữa 3 split",
              fontsize=12, fontweight="bold")
ax2.legend(fontsize=10)
ax2.grid(axis="y", alpha=0.3)

plt.tight_layout()

strat_fig_path = os.path.join(RESULT_OUTPUT_ROOT, "stratification_check.png")
plt.savefig(strat_fig_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig2)

print(f"\n✅ Biểu đồ phân phối split lưu tại: {split_fig_path}")
print(f"✅ Biểu đồ kiểm chứng stratification lưu tại: {strat_fig_path}")

In [ ]:
# ==============================================================================
# Cell 7: DATASET & DATA COLLATOR (UNIFORM 16-FRAME TEMPORAL SAMPLING) — FIXED
# ==============================================================================
import logging

dataset_logger = logging.getLogger("ViViTDataset")
dataset_logger.setLevel(logging.WARNING)
if not dataset_logger.handlers:
    dataset_logger.addHandler(logging.StreamHandler())


class ViViTVideoDataset(Dataset):
    def __init__(self, df: pd.DataFrame, processor, num_frames: int = 16,
                 is_train: bool = True, resize_short_edge: int = 224):
        self.df = df.reset_index(drop=True)
        self.processor = processor
        self.num_frames = num_frames
        self.is_train = is_train
        self.resize_hw = (resize_short_edge, resize_short_edge)

    def __len__(self):
        return len(self.df)

    def _sample_frame_indices(self, total_frames: int):
        """
        Trả về mảng ĐÚNG num_frames phần tử (có thể trùng index cho video ngắn).
        Không bao giờ dùng set() — giữ nguyên thứ tự và số lượng.
        """
        if total_frames <= 0:
            return np.zeros(self.num_frames, dtype=int)

        if total_frames <= self.num_frames:
            # Video ngắn: linspace tạo duplicate indices — hoàn toàn hợp lệ
            indices = np.linspace(0, total_frames - 1, self.num_frames)
        else:
            if self.is_train:
                # Phân đoạn ngẫu nhiên có trật tự thời gian
                segments = np.linspace(0, total_frames, self.num_frames + 1, dtype=int)
                indices = np.array([
                    random.randint(int(segments[i]), int(segments[i + 1]) - 1)
                    for i in range(self.num_frames)
                ])
            else:
                # Eval/Test: uniform sampling deterministic
                indices = np.linspace(0, total_frames - 1, self.num_frames)

        return np.clip(np.round(indices).astype(int), 0, max(total_frames - 1, 0))

    def _decode_video(self, video_path: str):
        """
        Decode video và trả về mảng LUÔN có shape (num_frames, H, W, 3).
        - Seek trực tiếp đến frame cần thiết (nhanh 10-100x cho video dài)
        - Fallback sang sequential nếu seek fail
        - Resize sớm để giảm RAM peak
        """
        cap = cv2.VideoCapture(video_path)
        if not cap.isOpened():
            dataset_logger.warning(
                f"[CORRUPT] Không mở được: {os.path.basename(video_path)}"
            )
            return np.zeros((self.num_frames, *self.resize_hw, 3), dtype=np.uint8)

        total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        if total_frames <= 0:
            cap.release()
            dataset_logger.warning(
                f"[CORRUPT] 0 frames: {os.path.basename(video_path)}"
            )
            return np.zeros((self.num_frames, *self.resize_hw, 3), dtype=np.uint8)

        # ── Lấy mảng đúng num_frames indices (CÓ THỂ TRÙNG cho video ngắn) ──
        full_indices = self._sample_frame_indices(total_frames)   # shape: (num_frames,)
        unique_indices = sorted(set(full_indices.tolist()))

        # ── Đọc video: seek trực tiếp đến frame cần thiết ──
        frames_dict = {}

        if total_frames > self.num_frames * 2:
            # Video dài: seek trực tiếp — tránh decode toàn bộ
            for target_idx in unique_indices:
                cap.set(cv2.CAP_PROP_POS_FRAMES, target_idx)
                ret, frame = cap.read()
                if ret:
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frame_rgb = cv2.resize(frame_rgb, self.resize_hw)
                    frames_dict[target_idx] = frame_rgb
        else:
            # Video ngắn: sequential (seek overhead > decode cost)
            unique_set = set(unique_indices)
            curr_idx = 0
            while True:
                ret, frame = cap.read()
                if not ret:
                    break
                if curr_idx in unique_set:
                    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
                    frame_rgb = cv2.resize(frame_rgb, self.resize_hw)
                    frames_dict[curr_idx] = frame_rgb
                    if len(frames_dict) == len(unique_set):
                        break
                curr_idx += 1

        cap.release()

        # ── FIX: Cảnh báo khi video bị truncated / đọc thiếu frame ──
        if len(frames_dict) < len(unique_indices):
            dataset_logger.warning(
                f"[TRUNCATED] Chỉ đọc {len(frames_dict)}/{len(unique_indices)} "
                f"unique frames: {os.path.basename(video_path)} (forward-fill applied)"
            )

        # ── Reconstruct mảng ĐÚNG num_frames frames từ full_indices ──
        fallback_frame = np.zeros((*self.resize_hw, 3), dtype=np.uint8)
        last_valid = fallback_frame

        frames = []
        for idx in full_indices:
            if idx in frames_dict:
                last_valid = frames_dict[idx]
            frames.append(last_valid)

        return np.stack(frames)  # LUÔN (num_frames, 224, 224, 3)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        video_path = row["filepath"]
        label = row["label_id"]

        frames = self._decode_video(video_path)  # (num_frames, 224, 224, 3)

        # Processor nhận list gồm num_frames frames ndarray
        inputs = self.processor(list(frames), return_tensors="pt")
        pixel_values = inputs["pixel_values"].squeeze(0)  # [num_frames, 3, 224, 224]

        return {
            "pixel_values": pixel_values,
            "labels": torch.tensor(label, dtype=torch.long),
            "video_path": video_path
        }


class ViViTDataCollator:
    """
    Collator chỉ trả tensor fields cho model.forward().
    Không truyền video_path vào batch dict — tránh Trainer đẩy strings lên GPU.
    """
    def __call__(self, batch):
        pixel_values = torch.stack([item["pixel_values"] for item in batch], dim=0)
        labels = torch.stack([item["labels"] for item in batch], dim=0)
        return {"pixel_values": pixel_values, "labels": labels}

In [ ]:
# ==============================================================================
# Cell 8: MODEL ARCHITECTURE DEFINITION — SUPPORTS CUSTOM NUM_FRAMES — FIXED
# ==============================================================================
import torch.nn.functional as F


class ViViTActionClassifier(nn.Module):
    """
    ViViT-Base backbone + custom classification head.
    Hỗ trợ num_frames khác pretrained (32) bằng interpolation positional embeddings.
    """

    def __init__(self, model_name: str, num_labels: int, num_frames: int = 16,
                 dropout_rate: float = 0.25, label_smoothing: float = 0.05,
                 _skip_pretrained: bool = False):
        super().__init__()

        # ── 1. Load backbone ──
        if _skip_pretrained:
            # FIX: load_pretrained dùng flag này để tránh download 2 lần
            from transformers import VivitConfig
            config = VivitConfig.from_pretrained(model_name)
            self.vivit = VivitModel(config)
        else:
            self.vivit = VivitModel.from_pretrained(model_name)

        self.config = self.vivit.config
        hidden_size = self.config.hidden_size

        # ── 2. Interpolate positional embeddings nếu num_frames khác pretrained ──
        pretrained_num_frames = self.config.num_frames  # 32
        if num_frames != pretrained_num_frames and not _skip_pretrained:
            self._interpolate_pos_embed(num_frames, pretrained_num_frames)
            self.config.num_frames = num_frames
            print(f"  ✅ Positional embeddings interpolated: {pretrained_num_frames} → {num_frames} frames")
            print(f"     Temporal tokens: {pretrained_num_frames // self.config.tubelet_size[0]}"
                  f" → {num_frames // self.config.tubelet_size[0]}")
        elif _skip_pretrained:
            self.config.num_frames = num_frames

        # ── 3. Config metadata ──
        self.config.num_labels = num_labels

        # ── 4. Classification head ──
        self.dropout = nn.Dropout(dropout_rate)
        self.classifier = nn.Linear(hidden_size, num_labels)
        nn.init.trunc_normal_(self.classifier.weight, std=0.02)
        nn.init.zeros_(self.classifier.bias)

        # ── 5. Loss ──
        self.loss_fn = nn.CrossEntropyLoss(label_smoothing=label_smoothing)

        # ── 6. Metadata cho serialization ──
        self._model_name = model_name
        self._num_labels = num_labels
        self._num_frames = num_frames
        self._dropout_rate = dropout_rate
        self._label_smoothing = label_smoothing

    def _interpolate_pos_embed(self, new_num_frames: int, old_num_frames: int):
        """
        FIX: Interpolate CHỈ temporal dimension, giữ nguyên spatial.
        Dùng F.interpolate 1D (mode='linear') trên trục temporal.

        Pretrained: 32 frames → 16 temporal tokens × 196 spatial = 3136 patches + 1 CLS
        New (16f) : 16 frames →  8 temporal tokens × 196 spatial = 1568 patches + 1 CLS
        """
        pos_embed = self.vivit.embeddings.position_embeddings.data  # (1, N_old+1, D)

        cls_embed = pos_embed[:, :1, :]    # (1, 1, D) — CLS token giữ nguyên
        patch_embed = pos_embed[:, 1:, :]  # (1, N_old, D)

        tubelet_t = self.config.tubelet_size[0]              # 2
        old_temporal = old_num_frames // tubelet_t            # 16
        new_temporal = new_num_frames // tubelet_t            # 8
        spatial_patches = patch_embed.shape[1] // old_temporal  # 196
        D = patch_embed.shape[-1]                             # 768

        # Reshape: (1, T_old × S, D) → (1, T_old, S, D)
        patch_embed = patch_embed.reshape(1, old_temporal, spatial_patches, D)

        # ── FIX: Interpolate 1D chỉ trên trục temporal ──
        # (1, T_old, S, D) → (S, D, T_old) cho F.interpolate 1D
        patch_embed = patch_embed.permute(0, 2, 3, 1).float()        # (1, S, D, T_old)
        patch_embed = patch_embed.reshape(spatial_patches, D, old_temporal)  # (S, D, T_old)

        patch_embed = F.interpolate(
            patch_embed,
            size=new_temporal,
            mode="linear",
            align_corners=False
        )  # (S, D, T_new)

        # Reshape ngược: (S, D, T_new) → (1, T_new, S, D) → (1, T_new × S, D)
        patch_embed = patch_embed.reshape(1, spatial_patches, D, new_temporal)
        patch_embed = patch_embed.permute(0, 3, 1, 2)   # (1, T_new, S, D)
        patch_embed = patch_embed.reshape(1, new_temporal * spatial_patches, D)

        # Ghép CLS + interpolated patches
        new_pos_embed = torch.cat([cls_embed, patch_embed], dim=1)
        self.vivit.embeddings.position_embeddings = nn.Parameter(new_pos_embed)

    # ──────────────────────────────────────────────────────────────────────
    def forward(self, pixel_values, labels=None, **kwargs):
        outputs = self.vivit(pixel_values=pixel_values)
        cls_output = outputs.last_hidden_state[:, 0, :]
        logits = self.classifier(self.dropout(cls_output))

        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)

        return ImageClassifierOutput(loss=loss, logits=logits)

    # ──────────────────────────────────────────────────────────────────────
    # Trong Cell 8:
    def gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None, **kwargs):
        try:
            self.vivit.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs=gradient_checkpointing_kwargs, **kwargs
            )
        except TypeError:
            self.vivit.gradient_checkpointing_enable(
                gradient_checkpointing_kwargs=gradient_checkpointing_kwargs
            )

    def gradient_checkpointing_disable(self):
        self.vivit.gradient_checkpointing_disable()

    @property
    def gradient_checkpointing(self):
        return getattr(self.vivit, "gradient_checkpointing", False)

    # ──────────────────────────────────────────────────────────────────────
    def save_pretrained(self, output_dir, **kwargs):
        os.makedirs(output_dir, exist_ok=True)
        torch.save(self.state_dict(), os.path.join(output_dir, "pytorch_model.bin"))
        self.config.save_pretrained(output_dir)
        arch_meta = {
            "model_name": self._model_name,
            "num_labels": self._num_labels,
            "num_frames": self._num_frames,
            "dropout_rate": self._dropout_rate,
            "label_smoothing": self._label_smoothing,
        }
        with open(os.path.join(output_dir, "architecture_config.json"), "w") as f:
            json.dump(arch_meta, f, indent=2)

    @classmethod
    def load_pretrained(cls, load_dir, device="cpu"):
        with open(os.path.join(load_dir, "architecture_config.json"), "r") as f:
            arch_meta = json.load(f)

        # FIX: _skip_pretrained=True → tránh download pretrained weights rồi ghi đè
        model = cls(
            model_name=arch_meta["model_name"],
            num_labels=arch_meta["num_labels"],
            num_frames=arch_meta.get("num_frames", 16),
            dropout_rate=arch_meta.get("dropout_rate", 0.25),
            label_smoothing=arch_meta.get("label_smoothing", 0.05),
            _skip_pretrained=True,
        )
        state_dict = torch.load(
            os.path.join(load_dir, "pytorch_model.bin"),
            map_location=device, weights_only=True
        )
        model.load_state_dict(state_dict)
        return model

In [ ]:
# ==============================================================================
# Cell 9: EVALUATION METRICS (Top@1, Top@3, Top@5 & Macro F1) — FIXED
# ==============================================================================

def topk_accuracy(logits, labels, k=5):
    """
    Vectorized Top-K accuracy — nhanh hơn ~10-50x so với list comprehension.
    """
    num_classes = logits.shape[1]
    k = min(k, num_classes)
    topk_preds = np.argsort(logits, axis=1)[:, -k:]  # (N, k) — top-k indices
    # Vectorized membership check: so sánh labels[:, None] với topk_preds
    match = np.any(topk_preds == labels[:, None], axis=1)  # (N,) bool
    return float(np.mean(match))


def compute_metrics(eval_pred):
    """
    Tính Top@1 Acc, Top@1 Macro F1, Top@3 Acc, Top@5 Acc.
    Guard an toàn cho trường hợp predictions là tuple.
    """
    logits, labels = eval_pred

    # ── Guard: Trainer có thể trả predictions dạng tuple ──
    if isinstance(logits, tuple):
        logits = logits[0]

    # ── Ép kiểu an toàn ──
    logits = np.asarray(logits, dtype=np.float32)
    labels = np.asarray(labels, dtype=np.int64)

    # ── Top-1 metrics ──
    top1_preds = np.argmax(logits, axis=-1)
    top1_acc = accuracy_score(labels, top1_preds)
    top1_macro_f1 = f1_score(labels, top1_preds, average="macro", zero_division=0)

    # ── Top-K accuracy (k=3 phân biệt hơn k=5 cho 16 class) ──
    top3_acc = topk_accuracy(logits, labels, k=3)
    top5_acc = topk_accuracy(logits, labels, k=5)

    return {
        "top1_macro_f1": top1_macro_f1,
        "top1_acc": top1_acc,
        "top3_acc": top3_acc,
        "top5_acc": top5_acc,
    }

In [ ]:
# ==============================================================================
# Cell 10: INSTANTIATE PROCESSOR & DATASETS — HARDENED
# ==============================================================================

# ── 1. Load Processor & Log Config ──
processor = VivitImageProcessor.from_pretrained(MODEL_NAME)

print(f"Processor config:")
print(f"  - Image Size  : {processor.size}")
print(f"  - Do Resize   : {processor.do_resize}")
print(f"  - Do Normalize: {processor.do_normalize}")
print(f"  - Image Mean  : {processor.image_mean}")
print(f"  - Image Std   : {processor.image_std}")
print(f"  - Resample    : {processor.resample}")

# ── 2. Verify Split Overlap (phòng thủ data leakage) ──
train_paths = set(train_df["filepath"])
val_paths = set(val_df["filepath"])
test_paths = set(test_df["filepath"])

overlap_tv = train_paths & val_paths
overlap_tt = train_paths & test_paths
overlap_vt = val_paths & test_paths

if overlap_tv or overlap_tt or overlap_vt:
    raise ValueError(
        f"DATA LEAKAGE DETECTED!\n"
        f"  Train ∩ Val : {len(overlap_tv)} files\n"
        f"  Train ∩ Test: {len(overlap_tt)} files\n"
        f"  Val ∩ Test  : {len(overlap_vt)} files"
    )
print("✅ Không có overlap giữa train/val/test splits.")

# ── 3. Verify File Existence ──
missing_files = []
for split_name, sdf in [("train", train_df), ("val", val_df), ("test", test_df)]:
    for fp in sdf["filepath"]:
        if not os.path.isfile(fp):
            missing_files.append((split_name, fp))

if missing_files:
    print(f"⚠️ {len(missing_files)} file không tồn tại trên disk:")
    for split_name, fp in missing_files[:5]:
        print(f"   [{split_name}] {fp}")
    raise FileNotFoundError(f"{len(missing_files)} video files not found. Kiểm tra DATASET_ROOT.")
print(f"✅ Tất cả {len(train_df) + len(val_df) + len(test_df)} file đều tồn tại.")

# ── 4. Instantiate Datasets ──
train_dataset = ViViTVideoDataset(train_df, processor, num_frames=NUM_FRAMES, is_train=True)
val_dataset = ViViTVideoDataset(val_df, processor, num_frames=NUM_FRAMES, is_train=False)
test_dataset = ViViTVideoDataset(test_df, processor, num_frames=NUM_FRAMES, is_train=False)

print(f"\nDataset đã sẵn sàng:")
print(f"  - Train: {len(train_dataset)} mẫu")
print(f"  - Val:   {len(val_dataset)} mẫu")
print(f"  - Test:  {len(test_dataset)} mẫu")

# ── 5. Sanity Check: Load 1 mẫu từ mỗi split ──
print("\nSanity check (load 1 mẫu từ mỗi split)...")
for ds_name, ds in [("Train", train_dataset), ("Val", val_dataset), ("Test", test_dataset)]:
    try:
        sample = ds[0]
        pv = sample["pixel_values"]
        lb = sample["labels"]
        assert pv.shape == (NUM_FRAMES, 3, 224, 224), \
            f"Shape sai: {pv.shape}, kỳ vọng ({NUM_FRAMES}, 3, 224, 224)"
        assert pv.dtype == torch.float32, f"Dtype sai: {pv.dtype}"
        assert 0 <= lb.item() < len(CLASSES), f"Label ngoài phạm vi: {lb.item()}"
        print(f"  ✅ {ds_name}: pixel_values={pv.shape}, label={lb.item()} ({CLASSES[lb.item()]})")
    except Exception as e:
        raise RuntimeError(f"❌ {ds_name} dataset lỗi khi load mẫu đầu tiên: {e}")

print("\n✅ Toàn bộ dataset sẵn sàng cho training.")

In [ ]:
# ==============================================================================
# Cell 11: SANITY CHECK - VISUALIZE TEMPORAL SAMPLING — FIXED
# ==============================================================================
sample_row = train_df.iloc[0]
sample_dataset_item = train_dataset[0]
frames_tensor = sample_dataset_item["pixel_values"]

actual_num_frames = frames_tensor.shape[0]

print(f"Kiểm tra mẫu đầu tiên:")
print(f"  - Đường dẫn: {sample_row['filepath']}")
print(f"  - Nhãn: {sample_row['class_name']} (ID: {sample_dataset_item['labels'].item()})")
print(f"  - Tensor Shape: {frames_tensor.shape}")
print(f"  - Expected frames: {NUM_FRAMES} | Actual: {actual_num_frames}")

if actual_num_frames != NUM_FRAMES:
    print(f"  ⚠️ MISMATCH! Video chỉ có {actual_num_frames} frames thay vì {NUM_FRAMES}.")
    print(f"     → Cần áp dụng fix Cell 7 (bỏ set() trên sampled_indices).")

# Adaptive: chọn 8 frame trải đều trên THỰC TẾ tensor size
n_show = min(8, actual_num_frames)
indices_to_show = np.linspace(0, actual_num_frames - 1, n_show, dtype=int).tolist()

proc_mean = torch.tensor(processor.image_mean).view(3, 1, 1)
proc_std = torch.tensor(processor.image_std).view(3, 1, 1)

def denormalize_frame(frame_chw):
    frame = frame_chw.cpu().float() * proc_std + proc_mean
    return frame.clamp(0.0, 1.0).permute(1, 2, 0).numpy()

cols = min(4, n_show)
rows = (n_show + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
if n_show == 1:
    axes = [axes]
else:
    axes = axes.flatten()

for i in range(n_show):
    frame_rgb = denormalize_frame(frames_tensor[indices_to_show[i]])
    axes[i].imshow(frame_rgb)
    axes[i].set_title(f"Frame: {indices_to_show[i]}/{actual_num_frames - 1}", fontsize=11)
    axes[i].axis("off")

# Ẩn axes thừa
for j in range(n_show, len(axes)):
    axes[j].axis("off")

fig.suptitle(
    f"Sanity Check — [{sample_row['class_name']}] — {actual_num_frames} frames",
    fontsize=14, fontweight="bold"
)
plt.tight_layout(rect=[0, 0, 1, 0.94])

sanity_fig_path = os.path.join(RESULT_OUTPUT_ROOT, "sanity_check_temporal_sampling.png")
plt.savefig(sanity_fig_path, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"\n✅ Figure lưu tại: {sanity_fig_path}")

In [ ]:
# ==============================================================================
# Cell 12: TRAINING ENGINE (AUTO-RESUME & CHECKPOINTING) — ROBUST & FIXED
# ==============================================================================
import inspect
from transformers.trainer_utils import get_last_checkpoint

# ── 0. HOTFIX TƯƠNG THÍCH GRADIENT CHECKPOINTING CHO CELL 8 TRÊN TRANSFORMERS MỚI ──
def _safe_gradient_checkpointing_enable(self, gradient_checkpointing_kwargs=None, **kwargs):
    """Tương thích với Transformers 4.47+ (hỗ trợ tham số every_n_layers)."""
    try:
        self.vivit.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs=gradient_checkpointing_kwargs, **kwargs
        )
    except TypeError:
        # Fallback cho phiên bản VivitModel cũ hơn không nhận every_n_layers
        self.vivit.gradient_checkpointing_enable(
            gradient_checkpointing_kwargs=gradient_checkpointing_kwargs
        )

# Gán method an toàn vào class
ViViTActionClassifier.gradient_checkpointing_enable = _safe_gradient_checkpointing_enable
ViViTActionClassifier.supports_gradient_checkpointing = True

run_name = "vivit_base_16frame_30epoch_16class"
run_output_dir = os.path.join(MODEL_OUTPUT_ROOT, run_name)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

# ── 1. Ước lượng training steps ──
n_gpus = max(torch.cuda.device_count(), 1)
steps_per_epoch = max(len(train_dataset) // (TRAIN_BATCH_SIZE * n_gpus * GRAD_ACCUM_STEPS), 1)
total_steps = steps_per_epoch * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

NUM_COSINE_CYCLES = max(NUM_EPOCHS // 10, 1)

print(f"{'='*70}")
print(f"TRAINING ESTIMATION")
print(f"{'='*70}")
print(f"  GPUs: {n_gpus} | Batch/GPU: {TRAIN_BATCH_SIZE} | Grad Accum: {GRAD_ACCUM_STEPS}")
print(f"  Effective batch size: {TRAIN_BATCH_SIZE * n_gpus * GRAD_ACCUM_STEPS}")
print(f"  Steps/epoch: ~{steps_per_epoch} | Total steps: ~{total_steps}")
print(f"  Warmup steps: ~{warmup_steps}")
print(f"  Precision: {'BF16' if use_bf16 else 'FP16'}")
print(f"  Eval/Save every: {EVAL_STEPS} steps (~{EVAL_STEPS / max(steps_per_epoch, 1):.1f} epochs)")

# ── 2. VRAM pre-check ──
if torch.cuda.is_available():
    for i in range(n_gpus):
        free_mem = torch.cuda.mem_get_info(i)[0] / 1024**3
        total_mem = torch.cuda.mem_get_info(i)[1] / 1024**3
        print(f"  [GPU {i}] Free VRAM: {free_mem:.1f} / {total_mem:.1f} GB")
        if free_mem < 10.0:
            print(f"  ⚠️ GPU {i}: VRAM thấp — có thể OOM với batch_size={TRAIN_BATCH_SIZE}")

# ── 3. Build TrainingArguments an toàn (Tương thích mọi phiên bản) ──
ta_params = inspect.signature(TrainingArguments.__init__).parameters

args_dict = {
    "output_dir": run_output_dir,
    "run_name": run_name,
    "remove_unused_columns": False,
    "seed": SEED,

    # Batch & Accumulation
    "per_device_train_batch_size": TRAIN_BATCH_SIZE,
    "per_device_eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRAD_ACCUM_STEPS,

    # Optimizer & Scheduler
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "lr_scheduler_type": "cosine_with_restarts",
    "num_train_epochs": NUM_EPOCHS,
    "warmup_steps": warmup_steps,

    # Logging & Saving
    "logging_steps": 25,
    "save_strategy": "steps",
    "eval_steps": EVAL_STEPS,
    "save_steps": SAVE_STEPS,

    # Best Model & Checkpointing
    "load_best_model_at_end": True,
    "metric_for_best_model": "top1_macro_f1",
    "greater_is_better": True,
    "save_total_limit": 3,

    # Precision & Memory
    "bf16": use_bf16,
    "fp16": torch.cuda.is_available() and not use_bf16,
    "gradient_checkpointing": True,

    # DataLoader
    "dataloader_num_workers": NUM_WORKERS,
    "dataloader_pin_memory": True,
    "report_to": "none",
}

if "eval_strategy" in ta_params:
    args_dict["eval_strategy"] = "steps"
else:
    args_dict["evaluation_strategy"] = "steps"

if "lr_scheduler_kwargs" in ta_params:
    args_dict["lr_scheduler_kwargs"] = {"num_cycles": NUM_COSINE_CYCLES}
else:
    args_dict["lr_scheduler_type"] = "cosine"

training_args = TrainingArguments(**args_dict)

# ── 4. Model ──
model = ViViTActionClassifier(
    MODEL_NAME,
    num_labels=len(CLASSES),
    num_frames=NUM_FRAMES,
    dropout_rate=0.25,
    label_smoothing=0.05
)

# ── 5. Trainer ──
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=ViViTDataCollator(),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE)],
)

# ── 6. Auto-Resume ──
last_ckpt = get_last_checkpoint(run_output_dir)
if last_ckpt:
    print(f"\n🔄 Phát hiện checkpoint cũ -> Tự động RESUME từ: {last_ckpt}")
else:
    print("\n🆕 Không có checkpoint cũ. Bắt đầu phiên huấn luyện mới.")

# ── 7. Train ──
train_result = trainer.train(resume_from_checkpoint=last_ckpt)

# ── 8. Persist training metrics ──
train_metrics = train_result.metrics
trainer.log_metrics("train", train_metrics)
trainer.save_metrics("train", train_metrics)

val_metrics = trainer.evaluate()
trainer.log_metrics("eval", val_metrics)
trainer.save_metrics("eval", val_metrics)

trainer.save_state()

print(f"\n{'='*70}")
print("Kết quả đánh giá trên tập Validation:")
for k, v in sorted(val_metrics.items()):
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")
print(f"{'='*70}")

In [ ]:
# ==============================================================================
# Cell 13: TEST SET INFERENCE & PERFORMANCE SUMMARY — FIXED
# ==============================================================================

# ── 1. Inference trên tập Test ──
pred_output = trainer.predict(test_dataset)   # Tự gọi model.eval()

# 2. Bóc tách an toàn predictions
logits = pred_output.predictions
if isinstance(logits, tuple):
    logits = logits[0]

logits = np.asarray(logits, dtype=np.float32)
test_labels = np.asarray(pred_output.label_ids, dtype=np.int64)

# ── 3. Metrics (tận dụng pred_output.metrics + bổ sung) ──
top1_preds = np.argmax(logits, axis=-1)
top1_acc = accuracy_score(test_labels, top1_preds)
top1_macro_f1 = f1_score(test_labels, top1_preds, average="macro", zero_division=0)
top3_acc = topk_accuracy(logits, test_labels, k=3)
top5_acc = topk_accuracy(logits, test_labels, k=5)

# ── 4. Classification Report đầy đủ ──
print("=" * 80)
print("CLASSIFICATION REPORT (TEST SET)")
print("=" * 80)
report_str = classification_report(
    test_labels, top1_preds,
    target_names=CLASSES,
    digits=4,
    zero_division=0
)
print(report_str)

# Lưu report text
report_path = os.path.join(RESULT_OUTPUT_ROOT, "vivit_classification_report.txt")
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report_str)

# ── 5. Bảng so sánh Val vs Test ──
summary_df = pd.DataFrame([{
    "Model": "ViViT-Base-16x2",
    "Val Top@1 Acc": val_metrics.get("eval_top1_acc"),
    "Val Top@1 F1": val_metrics.get("eval_top1_macro_f1"),
    "Val Top@3 Acc": val_metrics.get("eval_top3_acc"),
    "Val Top@5 Acc": val_metrics.get("eval_top5_acc"),
    "Test Top@1 Acc": top1_acc,
    "Test Top@1 F1": top1_macro_f1,
    "Test Top@3 Acc": top3_acc,
    "Test Top@5 Acc": top5_acc,
}])

metric_cols = [c for c in summary_df.columns if c != "Model"]
summary_df[metric_cols] = summary_df[metric_cols].round(4)

# ── 6. Phát hiện overfit ──
val_f1 = val_metrics.get("eval_top1_macro_f1")
if val_f1 is not None:
    gap = val_f1 - top1_macro_f1
    if gap > 0.05:
        print(f"⚠️ Overfitting gap: Val F1 ({val_f1:.4f}) - Test F1 ({top1_macro_f1:.4f}) = {gap:.4f}")
    elif gap < -0.02:
        print(f"ℹ️ Test cao hơn Val: gap = {gap:.4f} (có thể do variance nhỏ)")
    else:
        print(f"✅ Val-Test gap nhỏ: {gap:.4f} — model generalize tốt.")

display(summary_df)

# ── 7. Lưu kết quả ──
summary_csv_path = os.path.join(RESULT_OUTPUT_ROOT, "vivit_performance_summary.csv")
summary_df.to_csv(summary_csv_path, index=False)
print(f"\nĐã lưu bảng tổng kết: {summary_csv_path}")
print(f"Đã lưu classification report: {report_path}")

In [ ]:
# ==============================================================================
# Cell 14: CONFUSION MATRIX VISUALIZATION (DUAL-VIEW) — ENHANCED
# ==============================================================================

# ── 1. Tính toán ──
num_classes = len(CLASSES)
cm_raw = confusion_matrix(test_labels, top1_preds, labels=range(num_classes))
row_sums = cm_raw.sum(axis=1)
cm_norm = cm_raw.astype("float") / (row_sums[:, np.newaxis] + 1e-8)

fig, axes = plt.subplots(1, 2, figsize=(26, 11))

# ── 2. Subplot 1: Raw Counts ──
im1 = axes[0].imshow(cm_raw, interpolation="nearest", cmap="Blues")
axes[0].set_title("ViViT Confusion Matrix (Raw Counts)", fontsize=14, pad=15)
fig.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

# ── 3. Subplot 2: Normalized Recall % ──
im2 = axes[1].imshow(cm_norm, interpolation="nearest", cmap="Blues")
axes[1].set_title("ViViT Confusion Matrix (Normalized Recall %)", fontsize=14, pad=15)
fig.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

# ── 4. Annotate cells ──
tick_marks = np.arange(num_classes)
thresh_raw = cm_raw.max() / 2.0
thresh_norm = 0.5

for ax, cm, is_norm, thresh in zip(axes, [cm_raw, cm_norm], [False, True], [thresh_raw, thresh_norm]):
    ax.set_xticks(tick_marks)
    ax.set_yticks(tick_marks)
    ax.set_xticklabels(CLASSES, rotation=45, ha="right", fontsize=9)
    ax.set_yticklabels(CLASSES, fontsize=9)
    ax.set_ylabel("True Label", fontsize=11)
    ax.set_xlabel("Predicted Label", fontsize=11)

    for i in range(num_classes):
        for j in range(num_classes):
            is_diag = (i == j)

            # Format text: normalized view hiển thị cả % và count
            if is_norm:
                val_str = f"{cm[i, j]:.1%}\n({cm_raw[i, j]})"
            else:
                val_str = str(cm[i, j])

            # Text color: trắng trên nền đậm, đen trên nền nhạt
            text_color = "white" if cm[i, j] > thresh else "black"

            # Highlight diagonal: bold + viền
            fontweight = "bold" if is_diag else "normal"
            fontsize = 8.5 if is_diag else 7.5

            ax.text(j, i, val_str,
                    ha="center", va="center",
                    fontsize=fontsize, fontweight=fontweight,
                    color=text_color)

        # Vẽ viền cho ô diagonal
        rect = plt.Rectangle((i - 0.5, i - 0.5), 1, 1,
                              linewidth=2, edgecolor="#ff6600",
                              facecolor="none", linestyle="-")
        ax.add_patch(rect)

plt.tight_layout()

# ── 5. Lưu và hiển thị ──
cm_path = os.path.join(RESULT_OUTPUT_ROOT, "vivit_confusion_matrix_dual.png")
plt.savefig(cm_path, dpi=250, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"Đã xuất biểu đồ ma trận nhầm lẫn kép tại: {cm_path}")

In [ ]:
# ==============================================================================
# Cell 15: COMPREHENSIVE ERROR ANALYSIS & CONFIDENCE GAP — FIXED
# ==============================================================================

# ── 1. Tính xác suất ──
probabilities = torch.softmax(torch.as_tensor(logits, dtype=torch.float32), dim=1).numpy()
error_rows = []

test_df_records = test_dataset.df.reset_index(drop=True)
test_paths = test_df_records["filepath"].tolist()

assert len(test_paths) == len(test_labels), "Lệch số lượng giữa metadata và kết quả dự đoán!"

# ── 2. Verify prediction-metadata alignment (phòng multi-GPU reorder) ──
# FIX: Nếu mismatch → raise thay vì swap label (swap tạo inconsistency tệ hơn)
dataset_labels = test_df_records["label_id"].values
mismatched = int(np.sum(test_labels != dataset_labels))
if mismatched > 0:
    raise RuntimeError(
        f"{mismatched}/{len(test_labels)} labels lệch thứ tự giữa Trainer output và Dataset!\n"
        "Multi-GPU reorder detected — predictions và file paths không align.\n"
        "→ Chạy trainer.predict() trên single GPU hoặc debug DistributedSampler."
    )
print("✅ Prediction order khớp hoàn toàn với Dataset order.")

# ── 3. Map class → group (an toàn) ──
class_to_group = (
    df_clean.drop_duplicates("class_name")
    .set_index("class_name")["group"]
    .to_dict()
)

# ── 4. Quét chi tiết các trường hợp phân loại sai ──
for idx, (path, y_true, y_pred) in enumerate(zip(test_paths, test_labels, top1_preds)):
    if y_true == y_pred:
        continue

    p_pred = float(probabilities[idx, y_pred])
    p_true = float(probabilities[idx, y_true])
    c_gap = p_pred - p_true

    true_cls = CLASSES[y_true]
    pred_cls = CLASSES[y_pred]
    true_grp = class_to_group.get(true_cls, "Unknown")
    pred_grp = class_to_group.get(pred_cls, "Unknown")
    source_val = test_df_records.iloc[idx].get("source", "Unknown")

    top5_idx = np.argsort(probabilities[idx])[-5:][::-1]

    if c_gap >= 0.60:
        severity = "Critical (High Confidence Blunder)"
    elif c_gap >= 0.30:
        severity = "Moderate"
    else:
        severity = "Borderline (Low Margin)"

    error_rows.append({
        "sample_index": idx,
        "file_name": os.path.basename(path),
        "source": source_val,
        "true_group": true_grp,
        "pred_group": pred_grp,
        "cross_group_error": true_grp != pred_grp,
        "true_label": true_cls,
        "predicted_label": pred_cls,
        "predicted_confidence": round(p_pred, 4),
        "true_label_probability": round(p_true, 4),
        "confidence_gap": round(c_gap, 4),
        "severity": severity,
        "video_path": path,
        "top5_predictions": " | ".join(CLASSES[c] for c in top5_idx),
        "top5_probabilities": " | ".join(f"{probabilities[idx, c]:.4f}" for c in top5_idx)
    })

# ── 5. Đóng gói DataFrame ──
expected_columns = [
    "sample_index", "file_name", "source", "true_group", "pred_group",
    "cross_group_error", "true_label", "predicted_label", "predicted_confidence",
    "true_label_probability", "confidence_gap", "severity", "video_path",
    "top5_predictions", "top5_probabilities"
]
error_df = pd.DataFrame(error_rows, columns=expected_columns)

if not error_df.empty:
    error_df = error_df.sort_values("confidence_gap", ascending=False).reset_index(drop=True)

error_csv_path = os.path.join(RESULT_OUTPUT_ROOT, "vivit_misclassified_videos.csv")
error_df.to_csv(error_csv_path, index=False)

# ── 6. Báo cáo tổng quan ──
print("=" * 80)
print(f"BÁO CÁO PHÂN TÍCH LỖI DỰ ĐOÁN (TEST SPLIT)")
print("=" * 80)
print(f"Tổng số video đoán sai: {len(error_df)} / {len(test_labels)} ({len(error_df)/len(test_labels):.2%})")

if not error_df.empty:
    cross_group_cnt = int(error_df["cross_group_error"].sum())
    print(f"Lỗi nhầm giữa Violence <-> Non-Violence: {cross_group_cnt} ca "
          f"({cross_group_cnt/len(error_df):.1%} trong số lỗi)")

    print("\n--- Phân bố lỗi theo nguồn video ---")
    display(error_df["source"].value_counts().reset_index().rename(
        columns={"index": "Source", "source": "Source", "count": "Error Count"}))

    print("\n--- Top 10 cặp nhầm lẫn phổ biến nhất ---")
    confusion_pairs = (
        error_df.groupby(["true_label", "predicted_label", "cross_group_error"])
        .size().reset_index(name="count")
        .sort_values("count", ascending=False).head(10)
    )
    display(confusion_pairs)

    print("\n--- Top 5 ca sai lệch nghiêm trọng nhất ---")
    display(error_df[["file_name", "source", "true_label", "predicted_label",
                       "confidence_gap", "severity"]].head(5))

    # ── 7. Trực quan hóa phân phối confidence gap ──
    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    # [A] Histogram confidence gap
    axes[0].hist(error_df["confidence_gap"], bins=20, color="#e74c3c",
                 edgecolor="white", alpha=0.8)
    axes[0].axvline(0.30, color="orange", linestyle="--", label="Moderate threshold")
    axes[0].axvline(0.60, color="red", linestyle="--", label="Critical threshold")
    axes[0].set_xlabel("Confidence Gap (p_pred - p_true)")
    axes[0].set_ylabel("Số lượng")
    axes[0].set_title("Phân phối Confidence Gap", fontweight="bold")
    axes[0].legend()

    # [B] Severity breakdown
    severity_counts = error_df["severity"].value_counts()
    sev_colors = {"Critical (High Confidence Blunder)": "#e74c3c",
                  "Moderate": "#f39c12", "Borderline (Low Margin)": "#3498db"}
    colors = [sev_colors.get(s, "#999") for s in severity_counts.index]
    axes[1].barh(severity_counts.index, severity_counts.values, color=colors, edgecolor="white")
    axes[1].set_xlabel("Số lượng lỗi")
    axes[1].set_title("Phân bố mức độ nghiêm trọng", fontweight="bold")
    for bar, val in zip(axes[1].patches, severity_counts.values):
        axes[1].text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                     str(val), va="center", fontweight="bold")

    plt.tight_layout()
    gap_fig_path = os.path.join(RESULT_OUTPUT_ROOT, "error_confidence_gap.png")
    plt.savefig(gap_fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    plt.close(fig)
    print(f"\n✅ Biểu đồ confidence gap đã lưu tại: {gap_fig_path}")

else:
    print("🎉 Mô hình đạt độ chính xác 100% trên tập Test!")

print(f"Chi tiết toàn bộ video lỗi: {error_csv_path}")

In [ ]:
# ==============================================================================
# Cell 16: EXPORT PER-CLASS ERROR AUDIT & SUMMARY MANIFEST — FIXED
# ==============================================================================
import re

per_class_dir = os.path.join(RESULT_OUTPUT_ROOT, "per_class_errors")
os.makedirs(per_class_dir, exist_ok=True)

manifest_records = []

for cls in CLASSES:
    cls_id = label2id[cls]

    # 1. Chuẩn hóa tên file an toàn cho hệ thống tệp
    safe_cls_name = re.sub(r"[^a-zA-Z0-9_\-]", "_", cls)
    safe_cls_name = re.sub(r"_+", "_", safe_cls_name).strip("_")

    # 2. Thống kê TP / FP / FN
    true_mask = (test_labels == cls_id)
    pred_mask = (top1_preds == cls_id)

    total_test_samples = int(true_mask.sum())
    tp = int((true_mask & pred_mask).sum())
    fp = int((~true_mask & pred_mask).sum())
    fn = int((true_mask & ~pred_mask).sum())

    # 3. Tính precision, recall, F1 (an toàn chia 0)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0

    # 4. Lọc False Negatives (bỏ sót)
    cls_errors = error_df[error_df["true_label"] == cls] if not error_df.empty else pd.DataFrame()
    misclassified_count = len(cls_errors)

    # 5. Xác định class bị nhầm sang nhiều nhất
    if not cls_errors.empty:
        pred_vc = cls_errors["predicted_label"].value_counts()
        top_confused_with = pred_vc.index[0]
        top_confused_count = int(pred_vc.iloc[0])
        most_confused_pair = f"{top_confused_with} ({top_confused_count} lần)"

        output_file = os.path.join(per_class_dir, f"{safe_cls_name}_misclassified.csv")
        cls_errors.to_csv(output_file, index=False)
    else:
        most_confused_pair = "None"

    # 6. Đánh dấu class không có dữ liệu test
    if total_test_samples == 0:
        note = "⚠️ No test samples"
    elif recall >= 1.0 and precision >= 1.0:
        note = "✅ Perfect"
    elif recall < 0.70:
        note = "🔴 Low recall"
    elif precision < 0.70:
        note = "🟠 Low precision"
    else:
        note = ""

    manifest_records.append({
        "class_name": cls,
        "test_samples": total_test_samples,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1_score": round(f1, 4),
        "most_confused_with": most_confused_pair,
        "note": note,
    })

# 7. Đóng gói manifest
manifest_df = pd.DataFrame(manifest_records).sort_values("f1_score", ascending=True).reset_index(drop=True)
manifest_csv_path = os.path.join(RESULT_OUTPUT_ROOT, "per_class_summary_manifest.csv")
manifest_df.to_csv(manifest_csv_path, index=False)

print("=" * 80)
print("BẢNG TỔNG HỢP KIỂM TOÁN LỖI THEO TỪNG CLASS (PER-CLASS ERROR AUDIT)")
print("=" * 80)
display(manifest_df)

# ── 8. Trực quan hóa: Per-class Precision / Recall / F1 ──
fig, ax = plt.subplots(figsize=(14, 7))

class_names = manifest_df["class_name"].tolist()
y_pos = np.arange(len(class_names))
bar_h = 0.25

ax.barh(y_pos - bar_h, manifest_df["precision"], bar_h,
        label="Precision", color="#3498db", edgecolor="white")
ax.barh(y_pos, manifest_df["recall"], bar_h,
        label="Recall", color="#2ecc71", edgecolor="white")
ax.barh(y_pos + bar_h, manifest_df["f1_score"], bar_h,
        label="F1 Score", color="#e74c3c", edgecolor="white")

ax.set_yticks(y_pos)
ax.set_yticklabels(class_names, fontsize=10)
ax.set_xlabel("Score", fontsize=11)
ax.set_title("Per-Class Performance: Precision / Recall / F1", fontsize=13, fontweight="bold")
ax.legend(fontsize=10, loc="lower right")
ax.set_xlim(0, 1.05)
ax.axvline(0.80, color="gray", linestyle="--", alpha=0.5, label="80% baseline")
ax.grid(axis="x", alpha=0.3)

# Annotate F1 values
for i, f1_val in enumerate(manifest_df["f1_score"]):
    ax.text(f1_val + 0.01, i + bar_h, f"{f1_val:.2f}", va="center", fontsize=9, fontweight="bold")

plt.tight_layout()
perf_fig_path = os.path.join(RESULT_OUTPUT_ROOT, "per_class_precision_recall_f1.png")
plt.savefig(perf_fig_path, dpi=200, bbox_inches="tight")
plt.show()
plt.close(fig)

print(f"\n- Manifest tổng kết: {manifest_csv_path}")
print(f"- Biểu đồ per-class: {perf_fig_path}")
print(f"- Chi tiết từng class: {per_class_dir}")

In [ ]:
# ==============================================================================
# Cell 17: EXPORT FINAL MODEL & PROCESSOR — FIXED
# ==============================================================================
from transformers.modeling_utils import unwrap_model

final_model_dir = os.path.join(PERSISTENT_RUN_ROOT, "final_vivit_model")
os.makedirs(final_model_dir, exist_ok=True)

# ── 1. Unwrap DDP/DataParallel an toàn ──
actual_model = unwrap_model(trainer.model)

# ── 2. Đồng bộ metadata vào config ──
actual_model.config.id2label = {str(k): v for k, v in id2label.items()}  # JSON-safe keys
actual_model.config.label2id = label2id
actual_model.config.num_labels = len(CLASSES)

# ── 3. Lưu mô hình qua method save_pretrained (nếu có từ Cell 8 fix) ──
if hasattr(actual_model, "save_pretrained"):
    actual_model.save_pretrained(final_model_dir)
    print(f"✅ Model saved via save_pretrained → {final_model_dir}")
else:
    # Fallback: lưu thủ công
    torch.save(actual_model.state_dict(), os.path.join(final_model_dir, "pytorch_model.bin"))
    actual_model.config.save_pretrained(final_model_dir)
    print(f"✅ Model saved (fallback) → {final_model_dir}")

# ── 4. Chỉ rank 0 ghi processor và standalone checkpoint ──
if trainer.is_world_process_zero():
    processor.save_pretrained(final_model_dir)

    # Standalone checkpoint — dùng UNWRAPPED model, không có prefix module.
    standalone_ckpt_path = os.path.join(final_model_dir, "custom_vivit_checkpoint.pt")
    torch.save(
        {
            "state_dict": actual_model.state_dict(),   # FIX: unwrapped, không có module.
            "config": actual_model.config.to_dict(),
            "label2id": label2id,
            "id2label": {str(k): v for k, v in id2label.items()},
            "classes": CLASSES,
            "model_name": MODEL_NAME,
            "num_frames": NUM_FRAMES,
            "training_args": {
                "learning_rate": LEARNING_RATE,
                "num_epochs": NUM_EPOCHS,
                "batch_size_effective": TRAIN_BATCH_SIZE * GRAD_ACCUM_STEPS * max(torch.cuda.device_count(), 1),
                "weight_decay": WEIGHT_DECAY,
            }
        },
        standalone_ckpt_path
    )
    print(f"✅ Standalone checkpoint: {standalone_ckpt_path}")

    # ── 5. Round-trip verification ──
    print("\n🔍 Verifying checkpoint round-trip load...")
    try:
        ckpt = torch.load(standalone_ckpt_path, map_location="cpu", weights_only=True)
        test_model = ViViTActionClassifier(
            model_name=ckpt["model_name"],
            num_labels=len(ckpt["classes"]),
        )
        test_model.load_state_dict(ckpt["state_dict"])
        test_model.eval()

        # Quick forward pass check
        dummy_input = torch.zeros(1, NUM_FRAMES, 3, 224, 224)
        with torch.no_grad():
            out = test_model(pixel_values=dummy_input)
        assert out.logits.shape == (1, len(CLASSES)), \
            f"Output shape sai: {out.logits.shape}"

        print(f"   ✅ Round-trip OK — output shape: {out.logits.shape}")
        del test_model, ckpt
        torch.cuda.empty_cache()

    except Exception as e:
        print(f"   ❌ Round-trip FAILED: {e}")
        print("   → Kiểm tra lại state_dict keys hoặc architecture mismatch.")

# ── 6. Liệt kê toàn bộ files đã export ──
print(f"\n{'='*70}")
print("HOÀN THÀNH PIPELINE HUẤN LUYỆN, ĐÁNH GIÁ VÀ XUẤT BẢN")
print(f"{'='*70}")
print(f"📁 Model & Processor: {final_model_dir}")

if os.path.isdir(final_model_dir):
    for f in sorted(os.listdir(final_model_dir)):
        fpath = os.path.join(final_model_dir, f)
        size_mb = os.path.getsize(fpath) / 1024**2
        print(f"   {'📄' if os.path.isfile(fpath) else '📂'} {f} ({size_mb:.1f} MB)")

print(f"\n📊 Reports: {RESULT_OUTPUT_ROOT}")
print(f"{'='*70}")

In [ ]:
# ==============================================================================
# Cell 18: TEST SET EVALUATION — TOP-1 & TOP-5 ACCURACY (CHỈ TẬP TEST)
# ==============================================================================
import os
import glob
import json
import torch
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score

print("=" * 75)
print("🎯 BÁO CÁO ĐÁNH GIÁ TRÊN TẬP TEST ĐỘC LẬP (TEST SET ONLY)")
print("=" * 75)

def topk_accuracy(logits, labels, k=5):
    """Tính Top-K Accuracy vectorized."""
    num_classes = logits.shape[1]
    k = min(k, num_classes)
    topk_preds = np.argsort(logits, axis=1)[:, -k:]
    match = np.any(topk_preds == labels[:, None], axis=1)
    return float(np.mean(match))

test_top1 = None
test_top3 = None
test_top5 = None
test_f1   = None
source_info = None

# ── 1. KIỂM TRA TRONG RAM (NẾU ĐÃ CHẠY CELL 13 TRONG PHIÊN HIỆN TẠI) ──
if "top1_acc" in globals() and "top5_acc" in globals():
    test_top1 = top1_acc
    test_top3 = globals().get("top3_acc", None)
    test_top5 = top5_acc
    test_f1   = globals().get("top1_macro_f1", None)
    source_info = "Tính toán trực tiếp từ biến RAM của Cell 13"

elif "logits" in globals() and "test_labels" in globals():
    l_arr = np.asarray(logits[0] if isinstance(logits, tuple) else logits, dtype=np.float32)
    y_arr = np.asarray(test_labels, dtype=np.int64)
    test_top1 = topk_accuracy(l_arr, y_arr, k=1)
    test_top3 = topk_accuracy(l_arr, y_arr, k=3)
    test_top5 = topk_accuracy(l_arr, y_arr, k=5)
    test_f1   = f1_score(y_arr, np.argmax(l_arr, axis=-1), average="macro", zero_division=0)
    source_info = "Tính toán từ mảng Logits tập Test trong RAM"

# ── 2. NẾU RAM CHƯA CÓ: ĐỌC TỪ FILE CSV CỦA 'vivit 30' ──
if test_top1 is None:
    csv_candidates = glob.glob("/kaggle/input/**/vivit_performance_summary.csv", recursive=True) + \
                     glob.glob("/kaggle/working/**/vivit_performance_summary.csv", recursive=True)

    for csv_p in csv_candidates:
        try:
            df = pd.read_csv(csv_p)
            # Tìm chính xác các cột bắt đầu bằng 'Test'
            cols = {c.lower().strip(): c for c in df.columns}
            for c_low, c_orig in cols.items():
                if "test" in c_low and ("top1" in c_low or "top@1" in c_low) and "acc" in c_low:
                    val = df[c_orig].iloc[0]
                    if pd.notna(val): test_top1 = float(val)
                elif "test" in c_low and ("top5" in c_low or "top@5" in c_low) and "acc" in c_low:
                    val = df[c_orig].iloc[0]
                    if pd.notna(val): test_top5 = float(val)
                elif "test" in c_low and ("top3" in c_low or "top@3" in c_low) and "acc" in c_low:
                    val = df[c_orig].iloc[0]
                    if pd.notna(val): test_top3 = float(val)
                elif "test" in c_low and "f1" in c_low:
                    val = df[c_orig].iloc[0]
                    if pd.notna(val): test_f1 = float(val)

            if test_top1 is not None and test_top5 is not None:
                source_info = f"File {os.path.basename(csv_p)}"
                break
        except Exception:
            continue

# ── 3. NẾU VẪN CHƯA CÓ: TỰ ĐỘNG CHẠY INFERENCE TRỰC TIẾP TRÊN TEST_DATASET ──
if (test_top1 is None or test_top5 is None) and "test_dataset" in globals():
    print("ℹ️ Dữ liệu Test chưa có sẵn trong file summary. Đang chạy nhanh Inference trên tập Test...")
    
    # Dùng trainer nếu có sẵn, hoặc dùng model
    if "trainer" in globals():
        pred_out = trainer.predict(test_dataset)
        l_arr = pred_out.predictions[0] if isinstance(pred_out.predictions, tuple) else pred_out.predictions
        y_arr = pred_out.label_ids
    elif "model" in globals():
        from torch.utils.data import DataLoader
        model.eval()
        loader = DataLoader(test_dataset, batch_size=EVAL_BATCH_SIZE if "EVAL_BATCH_SIZE" in globals() else 16,
                            shuffle=False, collate_fn=ViViTDataCollator() if "ViViTDataCollator" in globals() else None)
        all_logits, all_labels = [], []
        with torch.no_grad():
            for batch in loader:
                p_val = batch["pixel_values"].cuda() if torch.cuda.is_available() else batch["pixel_values"]
                out = model(pixel_values=p_val)
                all_logits.append(out.logits.cpu().numpy())
                all_labels.append(batch["labels"].numpy())
        l_arr = np.concatenate(all_logits, axis=0)
        y_arr = np.concatenate(all_labels, axis=0)

    l_arr = np.asarray(l_arr, dtype=np.float32)
    y_arr = np.asarray(y_arr, dtype=np.int64)
    test_top1 = topk_accuracy(l_arr, y_arr, k=1)
    test_top3 = topk_accuracy(l_arr, y_arr, k=3)
    test_top5 = topk_accuracy(l_arr, y_arr, k=5)
    test_f1   = f1_score(y_arr, np.argmax(l_arr, axis=-1), average="macro", zero_division=0)
    source_info = "Inference trực tiếp trên test_dataset"

# ── 4. HIỂN THỊ KẾT QUẢ TEST SET ──
if test_top1 is not None and test_top5 is not None:
    print(f"📁 Nguồn trích xuất: {source_info}\n")
    print("┌" + "─" * 58 + "┐")
    print(f"│  ⭐ TEST TOP-1 ACCURACY : {test_top1:.4f}  ({test_top1 * 100:6.2f}%)       │")
    if test_top3 is not None:
        print(f"│     TEST TOP-3 ACCURACY : {test_top3:.4f}  ({test_top3 * 100:6.2f}%)       │")
    print(f"│  ⭐ TEST TOP-5 ACCURACY : {test_top5:.4f}  ({test_top5 * 100:6.2f}%)       │")
    if test_f1 is not None:
        print(f"│     TEST MACRO F1-SCORE : {test_f1:.4f}                       │")
    print("└" + "─" * 58 + "┘")
else:
    print("⚠️ Không thể trích xuất Top-1 và Top-5 của tập Test.")
    print("   Lý do: File `vivit_performance_summary.csv` trong output `vivit 30` chưa lưu kết quả test,")
    print("   đồng thời `test_dataset` chưa được load trong session hiện tại.")
    print("👉 Giải pháp: Hãy chạy Cell 10 (để khởi tạo `test_dataset`) rồi chạy lại Cell 18 này!")

print("=" * 75)